## Projet : Aide à la recherche de logement étudiant à Paris

### Contexte et persona

Le projet s’appuie sur le profil d’un **étudiant(e) cherchant un logement à Paris**, souhaitant trouver une **offre adaptée à son budget**,  
située **à proximité de son école** et bénéficiant d’un **environnement agréable** (transports, espaces verts, commerces, etc.).

---

###  Objectif du projet

L’objectif de ce projet est de concevoir une **solution interactive** d’aide à la décision permettant :
- d’**analyser et comparer les logements CROUS et privés**,  
- de **visualiser les établissements scolaires** et les **infrastructures environnantes**,  
- d’**évaluer l’accessibilité** (transports, parcs, commerces),  
- et de **recommander les logements optimaux** selon les préférences de l’utilisateur.

Chaque module (widget) répond à un besoin spécifique :
1. **Carte globale** : visualiser les écoles, logements et transports à proximité.  
2. **Espaces verts** : identifier la qualité de vie autour d’un logement.  
3. **Analyse des loyers CROUS/privés** : comparer les prix selon la surface et la localisation.  
4. **Commerces** : repérer la densité et le type de commerces disponibles.  
5. **Recommandation personnalisée** : trouver les meilleurs logements selon le budget, la distance et la taille.

7. Indice de Sécurité et Coût de Vie par Département
8. Score d'Accessibilité par École
9. Simulateur de Budget Mensuel Étudiant
10.Carte de Densité des Services Étudiants

---

###  Enjeux

Ce projet illustre la manière dont la **data visualisation** et les **outils interactifs Python (ipywidgets, Folium, Plotly)**  
peuvent être mobilisés pour créer une application d’aide à la décision dans le **domaine du logement étudiant**.  
Il combine **données géographiques, économiques et environnementales** afin de fournir une vision globale et personnalisée de l’offre à Paris.


Imports du projet


In [ ]:
# Importations nécessaires
import os
import re
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import folium
from folium.plugins import HeatMap, MarkerCluster
from geopy.distance import geodesic
from pyproj import Transformer
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, FloatSlider, IntSlider, VBox, Layout, HTML as WidgetHTML
from IPython.display import display, clear_output, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10


Chemins d'acces de tous nos fichiers


In [ ]:
PATH_MAG_CLASS = "data/magasins_classiques_long.csv"
PATH_MAG_FOOD  = "data/magasins_nourriture_long.csv"
PATH_PARKS     = "data/espaces_verts_idf.csv"
PATH_ECOLES    = "data/etablissements_IDF_final.csv"
PATH_CROUS     = "data/crous_idf_loyers_geo_complet.csv"
PATH_GARES     = "data/emplacement-des-gares-idf.csv"
PATH_NONCROUS  = "data/idf_lt50_appart_avec_prix_location.csv"
PATH_CRIME = "data/taux_criminalite_idf_par_departement_2024.csv"


# Chargement des données

crous = pd.read_csv(PATH_CROUS)
noncrous = pd.read_csv(PATH_NONCROUS)
ecoles = pd.read_csv(PATH_ECOLES)
gares = pd.read_csv(PATH_GARES, sep=';')
parks = pd.read_csv(PATH_PARKS)
mag_food = pd.read_csv(PATH_MAG_FOOD)
mag_class = pd.read_csv(PATH_MAG_CLASS)
crime = pd.read_csv(PATH_CRIME)

if not all(os.path.exists(p) for p in [PATH_ECOLES, PATH_CROUS, PATH_GARES, PATH_NONCROUS]):
    missing = [p for p in [PATH_ECOLES, PATH_CROUS, PATH_GARES, PATH_NONCROUS] if not os.path.exists(p)]
    raise FileNotFoundError(f" Fichiers manquants : {missing}")

In [ ]:
def tri_naturel(liste):
    def cle_naturelle(s):
        return [int(t) if t.isdigit() else t for t in re.split(r'(\d+)', str(s))]
    return sorted(liste, key=cle_naturelle)

def pick(df, candidates):
    """Retourne le nom réel de colonne correspondant au 1er alias trouvé (case-insensitif)."""
    m = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c and c.lower() in m:
            return m[c.lower()]
    return None

def smart_to_float(series):
    """Convertit proprement en float (gestion virgule, strings, None)."""
    return (
        series.astype(str)
              .str.replace(",", ".", regex=False)
              .str.extract(r"(-?[0-9]*\.?[0-9]+)")[0]
              .astype(float)
    )
def dist_km(lat1, lon1, lat2, lon2):
    """Calcul approximatif de distance en km."""
    return np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2) * 111

def extract_dept_from_cp(code_postal):
    """Extrait le département du code postal."""
    cp_str = str(code_postal).zfill(5)
    if cp_str.startswith('75'):
        return '75'
    return cp_str[:2]


### *Widget 1* :  Carte : écoles, transports et logements CROUS

Ce widget interactif permet de **visualiser la position d’une école** ainsi que son **environnement ** :  
les **résidences CROUS**, les **logements privés uniques les plus proches**, et les **gares accessibles à pied**.

L’utilisateur peut :
- Sélectionner une **école** dans la liste proposée,  
- Observer sur une **carte dynamique** :
  - En bleu,  L’emplacement de l’école sélectionnée,  
  - En violet,  Les **2 résidences CROUS** les plus proches,  
  - En vert, Les **8 logements privés uniques** les plus proches,  
  - En rouge,  Les **3 gares** les plus proches avec leur distance en mètres.

Chaque point sur la carte comporte une **Bulle d'informations** indiquant :
- Le **nom** ou le **type de lieu**,  
- L’**adresse complète**,  
- Et la **distance précise** depuis l’école.

Ce module offre une **vue géographique claire et complète** de l’environnement étudiant,  
en combinant l’offre de logements, les infrastructures de transport et la localisation des établissements d’enseignement.


In [ ]:
#  Détection des colonnes clés dynamiquement
COL_ENOM  = pick(ecoles, ["Nom de l'établissement","nom","libellé","etablissement","ecole"])
COL_EADDR = pick(ecoles, ["Adresse complète","Adresse"])
COL_ECP   = pick(ecoles, ["Code postal","cp"])
COL_ELAT  = pick(ecoles, ["latitude","lat"])
COL_ELON  = pick(ecoles, ["longitude","lon"])

COL_CLNOM  = pick(crous, ["Nom","résidence"])
COL_CLADDR = pick(crous, ["Adresse"])
COL_CLCP   = pick(crous, ["Code postal","cp"])
COL_CLLAT  = pick(crous, ["latitude","lat"])
COL_CLLON  = pick(crous, ["longitude","lon"])
COL_CLMED  = pick(crous, ["Loyer_median","loyer_median","loyer"])

COL_NNOM   = pick(noncrous, ["nom","name","adresse"])
COL_NADDR  = pick(noncrous, ["adresse"])
COL_NCP    = pick(noncrous, ["code_postal","cp"])
COL_NLAT   = pick(noncrous, ["latitude","lat"])
COL_NLON   = pick(noncrous, ["longitude","lon"])
COL_NSURF  = pick(noncrous, ["surface_m2","surface"])
COL_NPRIX  = pick(noncrous, ["prix","loyer","loyer_mensuel"])

for df, lat, lon in [(ecoles, COL_ELAT, COL_ELON), (crous, COL_CLLAT, COL_CLLON), (noncrous, COL_NLAT, COL_NLON)]:
    df[lat] = smart_to_float(df[lat])
    df[lon] = smart_to_float(df[lon])
    df.dropna(subset=[lat, lon], inplace=True)

# Conversion Lambert -> WGS84 pour gares
gares = gares.dropna(subset=['x','y']).copy()
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
gares['longitude'], gares['latitude'] = transformer.transform(gares['x'], gares['y'])

#  Widget
dropdown = Dropdown(
    options=sorted(ecoles[COL_ENOM].dropna().unique().tolist()),
    description='École :',
    style={'description_width': 'initial'},
    layout={'width': '460px'}
)

def afficher_carte(nom_ecole):
    row = ecoles[ecoles[COL_ENOM] == nom_ecole]
    if row.empty:
        print("Aucune école trouvée.")
        return

    e_lat, e_lon = float(row.iloc[0][COL_ELAT]), float(row.iloc[0][COL_ELON])
    e_addr = str(row.iloc[0].get(COL_EADDR, ""))

    #  Distances CROUS
    C = crous.copy()
    C['distance_km'] = C[[COL_CLLAT, COL_CLLON]].apply(
        lambda x: geodesic((e_lat, e_lon), (x[COL_CLLAT], x[COL_CLLON])).km, axis=1)
    C['distance_m'] = (C['distance_km'] * 1000).round().astype(int)
    crous_top = C.sort_values('distance_km').head(2).copy()

    #  Distances NON-CROUS
    N = noncrous.copy()
    N['distance_km'] = N[[COL_NLAT, COL_NLON]].apply(
        lambda x: geodesic((e_lat, e_lon), (x[COL_NLAT], x[COL_NLON])).km, axis=1)
    N['distance_m'] = (N['distance_km'] * 1000).round().astype(int)

    # Suppression des doublons (même adresse + même distance)
    N = N.drop_duplicates(subset=[COL_NADDR, 'distance_m'], keep='first')

    # Garde les 8 logements privés uniques les plus proches
    nc_top = N.sort_values('distance_km').head(8).copy()

    #  Gares proches
    G = gares.copy()
    G['distance_km'] = G.apply(lambda r: geodesic((e_lat, e_lon), (r['latitude'], r['longitude'])).km, axis=1)
    G['distance_m'] = (G['distance_km'] * 1000).round().astype(int)
    gares_proches = G.sort_values('distance_km').head(3).copy()

    #  Carte Folium
    m = folium.Map(location=(e_lat, e_lon), zoom_start=15)
    folium.Marker((e_lat, e_lon), popup=f"<b>{nom_ecole}</b><br>{e_addr}",
                  icon=folium.Icon(color='blue', icon='graduation-cap', prefix='fa')).add_to(m)

    #  Marqueurs CROUS
    for _, r in crous_top.iterrows():
        html = f"<b>(CROUS)</b> {r.get(COL_CLNOM, '')}<br>{r.get(COL_CLADDR,'')}<br>{r['distance_m']} m"
        folium.Marker((r[COL_CLLAT], r[COL_CLLON]),
                      popup=folium.Popup(html, max_width=320),
                      icon=folium.Icon(color='purple', icon='university', prefix='fa')).add_to(m)

    #  Marqueurs NON CROUS (uniques)
    for _, r in nc_top.iterrows():
        html = f"<b>(Non-CROUS)</b><br>{r.get(COL_NADDR,'')}<br>{r['distance_m']} m"
        folium.Marker((r[COL_NLAT], r[COL_NLON]),
                      popup=folium.Popup(html, max_width=320),
                      icon=folium.Icon(color='green', icon='home', prefix='fa')).add_to(m)

    #  Gares
    for _, r in gares_proches.iterrows():
        tooltip = f"{r['nom_long']} — {int(r['distance_m'])} m"
        folium.Marker((r['latitude'], r['longitude']),
                      tooltip=tooltip, icon=folium.Icon(color='red', icon='train', prefix='fa')).add_to(m)

    display(m)

# Affichage sans texte "afficher_carte"
from ipywidgets import interactive
from IPython.display import display

widget = interactive(afficher_carte, nom_ecole=dropdown)
display(widget)


### *Widget 2* :  Espaces verts à proximité des logements

Ce widget permet d’identifier les **espaces verts les plus proches** d’un **logement ** sélectionné.  
Il s’appuie sur les coordonnées géographiques (latitude et longitude) des logements et des parcs afin de calculer les distances réelles.

L’utilisateur peut :
- Choisir un **logement** dans la liste proposée,  
- Visualiser sur une **carte interactive** les **5 espaces verts les plus proches**,  
- Consulter pour chaque parc la **distance en kilomètres** depuis le logement choisi.

Les repères sur la carte sont distingués par couleur :
-  **Bleu** : position du logement sélectionné,  
-  **Vert** : emplacements des espaces verts à proximité.

Ce module permet d’évaluer rapidement la **qualité de l’environnement** d’un logement étudiant,  
en mettant en évidence la **proximité des zones vertes** et leur accessibilité depuis le lieu de résidence.


In [ ]:
#  Conversion des coordonnées
for df in [noncrous, parks]:
    for col in df.columns:
        if re.search("lat", col, re.I):
            df[col] = smart_to_float(df[col])
        if re.search("lon|lng", col, re.I):
            df[col] = smart_to_float(df[col])

noncrous.dropna(subset=[c for c in noncrous.columns if "lat" in c.lower() or "lon" in c.lower()], inplace=True)
parks.dropna(subset=[c for c in parks.columns if "lat" in c.lower() or "lon" in c.lower()], inplace=True)

#  Détection dynamique des colonnes
nom_col_n = [c for c in noncrous.columns if "nom" in c.lower() or "adresse" in c.lower()][0]
lat_n = [c for c in noncrous.columns if "lat" in c.lower()][0]
lon_n = [c for c in noncrous.columns if "lon" in c.lower()][0]
latp = [c for c in parks.columns if "lat" in c.lower()][0]
lonp = [c for c in parks.columns if "lon" in c.lower()][0]

#  Menu déroulant (tous les logements privés)
dropdown_logement = Dropdown(
    options=sorted(noncrous[nom_col_n].dropna().unique().tolist()),
    description='Logement :',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

#  Fonction principale
def espaces_verts_proches(nom_logement):
    row = noncrous[noncrous[nom_col_n] == nom_logement]
    if row.empty:
        return

    l_lat, l_lon = float(row.iloc[0][lat_n]), float(row.iloc[0][lon_n])

    # Calcul des distances avec les espaces verts
    parks["distance_km"] = parks[[latp, lonp]].apply(
        lambda x: geodesic((l_lat, l_lon), (x[latp], x[lonp])).km,
        axis=1
    )
    top_parks = parks.sort_values("distance_km").head(5)

    #Création de la carte
    m = folium.Map(location=[l_lat, l_lon], zoom_start=14)
    folium.Marker(
        [l_lat, l_lon],
        popup=f"Logement : {nom_logement}",
        icon=folium.Icon(color="blue", icon="home", prefix="fa")
    ).add_to(m)

    for _, r in top_parks.iterrows():
        nom_parc = r.get("nom", "Espace vert")
        dist = f"{r['distance_km']:.2f} km"
        folium.Marker(
            [r[latp], r[lonp]],
            popup=f"{nom_parc}<br>Distance : {dist}",
            icon=folium.Icon(color="green", icon="leaf", prefix="fa")
        ).add_to(m)

    #  Légende
    legend_html = """
    <div style="
        position: fixed;
        bottom: 25px; left: 25px; width: 160px;
        background-color: rgba(255,255,255,0.8);
        border-radius: 8px; padding: 8px; font-size: 12px;
        box-shadow: 0 0 5px rgba(0,0,0,0.3);
    ">
    <b>Légende</b><br>
    <span style="color:blue;">&#9679;</span> Logement<br>
    <span style="color:green;">&#9679;</span> Espaces verts
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    display(m)

_ = interact(espaces_verts_proches, nom_logement=dropdown_logement)


### *Widget 3* :  Répartition des écoles par secteur et département

Ce widget permet de visualiser les établissements scolaires en fonction de **leur secteur** (Public ou Privé)  
et de **leur localisation géographique**, identifiée grâce aux deux premiers chiffres du **code postal**.

L’utilisateur peut sélectionner :
- Un **département**  pour filtrer les écoles situées dans cette zone,  
- Un **secteur** pour n’afficher qu’un certain type d’établissement.

Le tableau interactif affiche ensuite la **liste des écoles correspondantes**, avec un **code couleur** :
-  **Bleu clair** pour les établissements **Publics**,  
-  **Orange clair** pour les établissements **Privés**.  


In [ ]:
# Harmonisation des colonnes
ecoles = ecoles.rename(columns={
    "Nom de l'établissement": "nom",
    "Secteur (Public/Privé)": "secteur",
    "Adresse": "adresse",
    "Code postal": "code_postal",
    "Ville": "ville",
    "Pays": "pays",
    "Adresse complète": "adresse_complete",
    "latitude": "lat",
    "longitude": "lon"
})

# Nettoyage de base
ecoles["secteur"] = ecoles["secteur"].str.strip().str.title()
ecoles["code_postal"] = ecoles["code_postal"].astype(str)
ecoles = ecoles.dropna(subset=["nom", "secteur", "code_postal"])

# Création des options de filtres
dep_options = ["(Tous)"] + sorted(ecoles["code_postal"].str[:2].unique().tolist())
secteurs = ["(Tous)"] + sorted(ecoles["secteur"].unique().tolist())

# Widgets
widgets_dep = widgets.Dropdown(options=dep_options, value="(Tous)", description="Département :")
widgets_secteur = widgets.Dropdown(options=secteurs, value="(Tous)", description="Secteur :")
widgets_bouton = widgets.Button(description="Afficher", button_style="info", icon="school")
out_ecoles = widgets.Output()

# Fonction principale d’affichage
def afficher_ecoles(b):
    with out_ecoles:
        clear_output(wait=True)

        data = ecoles.copy()

        # Filtre par département (2 premiers chiffres du code postal)
        if widgets_dep.value != "(Tous)":
            data = data[data["code_postal"].str.startswith(widgets_dep.value)]

        # Filtre par secteur
        if widgets_secteur.value != "(Tous)":
            data = data[data["secteur"] == widgets_secteur.value]

        if data.empty:
            display(HTML("<b style='color:red;'>Aucune école trouvée pour ces critères.</b>"))
            return

        #  Tableau coloré
        def colorer_secteur(val):
            if val == "Public":
                return "background-color: #BBDEFB; color: #0D47A1;"
            elif val == "Privé":
                return "background-color: #FFE0B2; color: #E65100;"
            else:
                return ""

        table = (
            data[["nom", "secteur", "ville", "code_postal"]]
            .sort_values("secteur")
            .head(25)
            .style.applymap(colorer_secteur, subset=["secteur"])
            .set_table_attributes("style='border-collapse:collapse; width:100%; font-size:14px;'")
        )

        display(HTML(f"<h3> {len(data)} écoles trouvées</h3>"))
        display(table)

# Liaison du bouton à la fonction
widgets_bouton.on_click(afficher_ecoles)

# Interface utilisateur finale
ui_ecoles = widgets.VBox([
    widgets.HTML("<h2> Répartition des écoles par secteur et département</h2>"
                 "<p>Choisis un département (via le code postal) et filtre selon le secteur "
                 "pour afficher la liste colorée des établissements.</p>"),
    widgets.HBox([widgets_dep, widgets_secteur]),
    widgets_bouton,
    out_ecoles
])

display(ui_ecoles)


### *Widget 4* :  Recommandation de logements autour des écoles

Ce widget interactif a pour objectif de **recommander les logements les plus pertinents** autour d’une école sélectionnée,  
en combinant à la fois les logements **CROUS** et **non-CROUS (privés)**.

L’utilisateur peut choisir :
- Une **école** parmi la liste proposée,  
- Un **budget maximum (€)**,  
- Une **surface minimale (m²)**,  
- Et une **distance maximale (km)** entre le logement et l’école.

L’algorithme calcule ensuite un **score de pertinence (/5)** pour chaque logement, basé sur :
- Sa **proximité avec l’école**,  
- Son **prix** par rapport au budget défini.  

Les résultats affichent les **10 meilleurs logements uniques**, avec pour chacun :
- Le **type de logement** (CROUS ou privé),  
- Le **nom** et l’**adresse**,  
- Le **prix estimé** et la **surface**,  
- La **distance à l’école** et au **centre de Paris**,  
- Et le **score global (/5)** visualisé sous forme de barre colorée.

Ce module permet ainsi de repérer rapidement les **meilleures opportunités de logement étudiant**,  
en prenant en compte les contraintes de budget, de surface et de localisation.


In [ ]:
PARIS_CENTER = (48.853, 2.35)

def dist_km(lat1, lon1, lat2, lon2):
    return np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2) * 111

for df in [ecoles, noncrous, crous]:
    for col in df.columns:
        if re.search("lat", col, re.I): df[col] = smart_to_float(df[col])
        if re.search("lon|lng", col, re.I): df[col] = smart_to_float(df[col])

ecole_col = [c for c in ecoles.columns if "nom" in c.lower()][0]
lat_e = [c for c in ecoles.columns if "lat" in c.lower()][0]
lon_e = [c for c in ecoles.columns if "lon" in c.lower()][0]

lat_n = [c for c in noncrous.columns if "lat" in c.lower()][0]
lon_n = [c for c in noncrous.columns if "lon" in c.lower()][0]
prix_col_n = "prix_estime"
surf_col_n = next((c for c in noncrous.columns if re.search("surface", c, re.I)), None)
addr_col_n = next((c for c in noncrous.columns if re.search("adresse|voie", c, re.I)), None)

lat_c = [c for c in crous.columns if "lat" in c.lower()][0]
lon_c = [c for c in crous.columns if "lon" in c.lower()][0]
prix_col_c = next((c for c in crous.columns if re.search("loyer|prix", c, re.I)), None)
surf_col_c = next((c for c in crous.columns if re.search("surface", c, re.I)), None)
nom_col_c = next((c for c in crous.columns if re.search("nom|résidence", c, re.I)), None)
addr_col_c = next((c for c in crous.columns if re.search("adresse", c, re.I)), None)

def recommander(ecole, budget_max, surface_min, distance_max):
    """Recommande 10 logements uniques selon les critères utilisateur."""
    row = ecoles[ecoles[ecole_col] == ecole]
    if row.empty:
        return
    e_lat, e_lon = float(row.iloc[0][lat_e]), float(row.iloc[0][lon_e])

    # --- Non-CROUS ---
    L = noncrous.copy()
    L["prix"] = smart_to_float(L[prix_col_n])
    L["surface"] = smart_to_float(L[surf_col_n]) if surf_col_n else np.nan
    L["distance_ecole_km"] = dist_km(L[lat_n], L[lon_n], e_lat, e_lon)
    L["distance_paris_km"] = dist_km(L[lat_n], L[lon_n], *PARIS_CENTER)
    L["type"] = "Non-CROUS"
    L["nom_logement"] = "Logement privé"
    L["adresse"] = L[addr_col_n] if addr_col_n else np.nan

    # --- CROUS ---
    C = crous.copy()
    C["prix"] = smart_to_float(C[prix_col_c])
    C["surface"] = smart_to_float(C[surf_col_c]) if surf_col_c else np.nan
    C["distance_ecole_km"] = dist_km(C[lat_c], C[lon_c], e_lat, e_lon)
    C["distance_paris_km"] = dist_km(C[lat_c], C[lon_c], *PARIS_CENTER)
    C["type"] = "CROUS"
    C["nom_logement"] = C[nom_col_c] if nom_col_c else "Résidence CROUS"
    C["adresse"] = C[addr_col_c] if addr_col_c else ""

    all_logs = pd.concat([L, C], ignore_index=True)
    all_logs = all_logs.dropna(subset=["prix"])
    all_logs = all_logs[all_logs["prix"] > 50]
    all_logs = all_logs[all_logs["prix"] <= budget_max]
    all_logs = all_logs[all_logs["surface"].fillna(0) >= surface_min]
    all_logs = all_logs[all_logs["distance_ecole_km"] <= distance_max]
    all_logs = all_logs.drop_duplicates(subset=["adresse", "prix", "type", "surface"], keep="first")

    if all_logs.empty:
        display(HTML(f"<p style='color:#ef4444;font-weight:bold;'>Aucun logement unique trouvé.</p>"))
        return

    all_logs["score_raw"] = (
        0.6 * (1 / (1 + all_logs["distance_ecole_km"])) +
        0.4 * (1 / (1 + (all_logs["prix"] / budget_max)))
    )
    all_logs["score"] = (all_logs["score_raw"] / all_logs["score_raw"].max()) * 5
    top10 = all_logs.sort_values("score", ascending=False).head(10)

    styled_df = (
        top10[["type", "nom_logement", "adresse", "prix", "surface",
               "distance_ecole_km", "distance_paris_km", "score"]]
        .rename(columns={
            "type": "Type",
            "nom_logement": "Nom du logement",
            "adresse": "Adresse",
            "prix": "Prix (€)",
            "surface": "Surface (m²)",
            "distance_ecole_km": "Distance école (km)",
            "distance_paris_km": "Distance Paris (km)",
            "score": "Score (/5)"
        })
        .style
        .set_caption(f"Logements uniques autour de {ecole}")
        .bar(subset=["Score (/5)"], color="#3b82f6")
    )
    display(styled_df)

interact(
    recommander,
    ecole=Dropdown(
        options=sorted(ecoles[ecole_col].dropna().unique().tolist()),
        description=" École :",
        style={'description_width': 'initial'},
        layout={'width': '400px'}
    ),
    budget_max=FloatSlider(value=800, min=300, max=2000, step=50, description=" Budget max (€)"),
    surface_min=FloatSlider(value=15, min=5, max=60, step=1, description=" Surface min (m²)"),
    distance_max=FloatSlider(value=3, min=1, max=10, step=0.5, description=" Distance max école (km)")
)


### *Widget 5* :  Magasins par arrondissement

Ce widget interactif permet d’explorer les **commerces ** selon deux grandes catégories :
- **Commerces de nourriture** (boulangeries, supermarchés, restaurants, etc.),
- **Commerces classiques** (habillement, électronique, librairies, etc.).

L’utilisateur peut successivement filtrer les données par :
- **Département**,  
- **Commune**,  
- Et, au choix :
  - Afficher la **répartition complète** des types de commerces dans cette zone,  
  - Ou sélectionner un **type de commerce spécifique** (par exemple : “Boulangerie” ou “Pharmacie”).

Le tableau de résultats affiche :
- Le **nombre total de commerces** recensés dans la commune choisie,  
- La **liste des types de commerces** avec leur nombre respectif,  
- Ou, en cas de filtrage précis, le **nombre d’établissements** correspondant au type choisi.

Ce module offre ainsi une **vue synthétique et personnalisable** de la répartition commerciale régionale, utile pour analyser l’offre de services sur un territoire.


In [ ]:
#   Widgets
choix_categorie = widgets.ToggleButtons(
    options=[" Nourriture", " Classique"],
    description="Catégorie :",
    button_style='info'
)

departement_widget = widgets.Dropdown(
    description=" Département :",
    options=[],
    style={'description_width': 'initial'}
)

commune_widget = widgets.Dropdown(
    description=" Commune :",
    options=[],
    style={'description_width': 'initial'}
)

btn_afficher_tout = widgets.Button(
    description="Afficher tout",
    button_style='success',
    icon="eye"
)
btn_choisir_categorie = widgets.Button(
    description="Choisir par catégorie",
    button_style='info',
    icon="filter"
)

type_magasin_widget = widgets.Dropdown(
    description=" Type de magasin :",
    options=[],
    style={'description_width': 'initial'}
)

#   Conteneurs
output = widgets.Output()
box_departement = widgets.VBox([])
box_commune = widgets.VBox([])
box_choix = widgets.VBox([])
box_type = widgets.VBox([])
box_result = widgets.VBox([])

#   Fonctions utilitaires
def clear_after(widget):
    """Efface les conteneurs après un changement"""
    for w in [box_commune, box_choix, box_type, box_result]:
        w.children = []
    output.clear_output()

def on_categorie_change(change):
    clear_after(choix_categorie)
    if change.new:
        df = mag_food if "Nourriture" in change.new else mag_class
        deps = sorted(df["departement"].dropna().unique())
        departement_widget.options = deps
        box_departement.children = [departement_widget]

def on_departement_change(change):
    clear_after(departement_widget)
    if change.new:
        df = mag_food if "Nourriture" in choix_categorie.value else mag_class
        communes = sorted(df[df["departement"] == change.new]["libelle_de_commune"].dropna().unique())
        commune_widget.options = communes
        box_commune.children = [commune_widget]

def on_commune_change(change):
    clear_after(commune_widget)
    if change.new:
        box_choix.children = [widgets.HBox([btn_afficher_tout, btn_choisir_categorie])]

#   Affichage principal sans graphique
def on_afficher_tout_clicked(b):
    box_type.children = []
    box_result.children = []
    output.clear_output()

    df = mag_food if "Nourriture" in choix_categorie.value else mag_class
    subset = df.copy()

    if "departement" in df.columns:
        subset = subset[subset["departement"] == departement_widget.value]
    if commune_widget.value:
        subset = subset[subset["libelle_de_commune"] == commune_widget.value]

    total = subset["nombre"].sum()
    types_counts = subset.groupby("type")["nombre"].sum().sort_values(ascending=False)

    with output:
        clear_output()
        display(HTML(f"""
        <div style='font-size:16px; line-height:1.6em'>
        <b> Département :</b> {departement_widget.value}<br>
        <b> Commune :</b> {commune_widget.value}<br>
        <b> Nombre total de commerces :</b>
        <span style='color:#00897B;font-weight:bold'>{total}</span>
        </div><br>
        """))

        if not types_counts.empty:
            rep_html = "<ul style='list-style:none; padding-left:10px;'>"
            for t, c in types_counts.items():
                rep_html += f"<li> <b>{t.replace('_', ' ').title()}</b> : <span style='color:#1976D2'>{c}</span></li>"
            rep_html += "</ul>"
            display(HTML("<h4> Répartition par type :</h4>" + rep_html))

    box_result.children = [output]

#   Choix par type de magasin
def on_choisir_categorie_clicked(b):
    box_result.children = []
    output.clear_output()

    df = mag_food if "Nourriture" in choix_categorie.value else mag_class
    types = sorted(df["type"].unique())
    type_magasin_widget.options = types
    box_type.children = [type_magasin_widget]

def on_type_change(change):
    box_result.children = []
    output.clear_output()

    if change.new:
        df = mag_food if "Nourriture" in choix_categorie.value else mag_class
        subset = df.copy()

        if "departement" in df.columns:
            subset = subset[subset["departement"] == departement_widget.value]
        if commune_widget.value:
            subset = subset[subset["libelle_de_commune"] == commune_widget.value]
        if type_magasin_widget.value:
            subset = subset[subset["type"] == type_magasin_widget.value]

        nb = subset["nombre"].sum()

        display(HTML(f"""
        <div style='font-size:16px; line-height:1.6em'>
        <b> Département :</b> {departement_widget.value}<br>
        <b> Commune :</b> {commune_widget.value}<br>
        <b> Type de magasin :</b> {type_magasin_widget.value.replace('_', ' ').title()}<br>
        <b> Nombre de commerces trouvés :</b>
        <span style='color:#00897B;font-weight:bold'>{nb}</span>
        </div>
        """))

#  Connexions
choix_categorie.observe(on_categorie_change, names='value')
departement_widget.observe(on_departement_change, names='value')
commune_widget.observe(on_commune_change, names='value')
type_magasin_widget.observe(on_type_change, names='value')
btn_afficher_tout.on_click(on_afficher_tout_clicked)
btn_choisir_categorie.on_click(on_choisir_categorie_clicked)

#   Interface finale
ui = widgets.VBox([
    widgets.HTML("<h3 style='color:#1565C0'>Sélection des commerces en Île-de-France</h3>"
                 "<p>Ce module permet de filtrer les commerces selon leur catégorie, département et commune, "
                 "et d’afficher la répartition ou les types de magasins disponibles.</p>"),
    choix_categorie,
    box_departement,
    box_commune,
    box_choix,
    box_type,
    box_result
])

display(ui)


### *Widget 6* : Comparaison des loyers — Logements CROUS et privés

Ce widget permet de comparer les **logements étudiants CROUS** avec les **logements privés** en Île-de-France.  
L’objectif est d’analyser la répartition des loyers et des surfaces disponibles selon plusieurs critères.

L’utilisateur peut filtrer les données selon :
- **La source** du logement → CROUS ou Privé,  
- **L’arrondissement** ,  
- **Le type** d’hébergement (appartement, studio, etc.),  
- Et la **surface** souhaitée (de 8 à 50 m²).

Le tableau interactif affiche :
- Les **10 premiers logements** correspondant aux filtres choisis,  
- La **surface moyenne**, le **loyer moyen** et la **médiane** pour la sélection effectuée.

Ce module permet ainsi d’identifier les tendances de prix entre les logements **publics (CROUS)** et **privés**,  
et d’aider à repérer les options les plus avantageuses selon la localisation et le type d’hébergement.


In [ ]:
crous_liste = []
for _, row in crous.iterrows():
    for i in range(1, 7):
        t = row.get(f"logement_{i}_type")
        s = row.get(f"logement_{i}_surface_m2")
        l = row.get(f"logement_{i}_loyer_eur")
        if pd.notna(s) and pd.notna(l) and s > 5:
            crous_liste.append({
                "source": "CROUS",
                "nom": row.get("nom", "Résidence CROUS"),
                "adresse": row.get("adresse", ""),
                "surface": float(s),
                "loyer": float(l),
                "type": t if pd.notna(t) else "Inconnu",
                "arrondissement": row.get("arrondissement", "")
            })
crous_clean = pd.DataFrame(crous_liste)

# Harmonisation des colonnes
noncrous = noncrous.rename(columns={
    "adresse": "adresse",
    "code_postal": "arrondissement",
    "surface_m2": "surface",
    "prix_estime": "loyer"
})
noncrous["surface"] = pd.to_numeric(noncrous["surface"], errors="coerce")
noncrous["loyer"] = pd.to_numeric(noncrous["loyer"], errors="coerce")

# Nettoyage basique des aberrations
nb_avant = len(noncrous)
noncrous = noncrous[(noncrous["surface"] >= 8) & (noncrous["surface"] <= 80)]
noncrous = noncrous[(noncrous["loyer"] > 100) & (noncrous["loyer"] < 5000)]
nb_apres = len(noncrous)

# Finitions
noncrous["source"] = "Privé"
noncrous["nom"] = "Logement privé"
noncrous["type"] = noncrous.get("type", pd.Series(["Appartement"] * len(noncrous)))
noncrous["type"] = noncrous["type"].fillna("Appartement")

noncrous = noncrous[["source", "nom", "adresse", "surface", "loyer", "type", "arrondissement"]]

# Fusion
df_all = pd.concat([crous_clean, noncrous], ignore_index=True)

# Typages finaux
df_all["arrondissement"] = df_all["arrondissement"].astype(str)
df_all["type"] = df_all["type"].fillna("Inconnu").astype(str)
df_all["adresse"] = df_all["adresse"].fillna("").astype(str)
df_all["nom"] = df_all["nom"].fillna("").astype(str)

# Widgets
sources = ["(Tous)", "CROUS", "Privé"]
arr_options = ["(Tous)"] + sorted(df_all["arrondissement"].dropna().astype(str).unique().tolist())
type_options = ["(Tous)"] + sorted(df_all["type"].dropna().astype(str).unique().tolist())

widgets_source = widgets.ToggleButtons(
    options=sources,
    description="Source :"
)
widgets_arr = widgets.Dropdown(
    options=arr_options, value="(Tous)", description="Arrondissement :"
)
widgets_type = widgets.Dropdown(
    options=type_options, value="(Tous)", description="Type :"
)

# Limite surface max = 50 m²
min_surface_raw = float(df_all["surface"].min()) if len(df_all) else 8.0
min_surface = max(8.0, min(min_surface_raw, 50.0))
max_surface = 50.0

widgets_surface = widgets.FloatRangeSlider(
    value=[min_surface, max_surface],
    min=min_surface, max=max_surface, step=0.5,
    description="Surface (m²) :", layout=widgets.Layout(width="90%")
)

widgets_afficher = widgets.Button(description="Afficher la liste")
out_recherche = widgets.Output()

# Fonction d'affichage
def afficher_resultats(_):
    with out_recherche:
        clear_output(wait=True)

        data = df_all.copy()
        if widgets_source.value != "(Tous)":
            data = data[data["source"] == widgets_source.value]
        if widgets_arr.value != "(Tous)":
            data = data[data["arrondissement"] == widgets_arr.value]
        if widgets_type.value != "(Tous)":
            data = data[data["type"] == widgets_type.value]
        data = data[
            (data["surface"] >= widgets_surface.value[0]) &
            (data["surface"] <= widgets_surface.value[1])
        ]

        if data.empty:
            display(HTML("<b style='color:#c00;'>Aucun logement trouvé pour ces critères.</b>"))
            return

        moy = data["loyer"].mean()
        med = data["loyer"].median()

        display(HTML(f"""
        <h3>Liste des logements disponibles</h3>
        <div style='font-size:15px; line-height:1.6em'>
        Source : <b>{widgets_source.value}</b><br>
        Arrondissement : <b>{widgets_arr.value}</b><br>
        Surface : <b>{widgets_surface.value[0]}–{widgets_surface.value[1]} m²</b><br>
        Loyer moyen : <b>{moy:.0f} €</b> — Médiane : <b>{med:.0f} €</b>
        </div>
        """))

        colonnes = ["source", "nom", "adresse", "surface", "loyer", "type", "arrondissement"]
        table = (data[colonnes]
                 .sort_values("loyer", ascending=True)
                 .reset_index(drop=True)
                 .head(10))
        table_html = table.to_html(index=False, border=0).replace(
            "<table", "<table style='border-collapse:collapse; width:100%; font-size:14px;'"
        ).replace(
            "<th>", "<th style='background:#1565C0; color:white; padding:6px;'>"
        ).replace(
            "<td>", "<td style='padding:6px; border-bottom:1px solid #ddd;'>"
        )

        display(HTML(f"<h4>{len(data)} logement(s) trouvé(s) — affichage des 10 premiers</h4>"))
        display(HTML(table_html))

widgets_afficher.on_click(afficher_resultats)

# Interface
ui = widgets.VBox([
    widgets.HTML("<h2>Comparaison des loyers — CROUS & logements privés</h2>"
                 "<p>Filtre par source, arrondissement, type et surface (jusqu’à 50 m²).</p>"),
    widgets.HBox([widgets_source, widgets_arr, widgets_type]),
    widgets_surface,
    widgets_afficher,
    out_recherche
])

display(ui)

# Affichage initial
afficher_resultats(None)


---
## *Widget 7* : Indice de Sécurité et Coût de Vie par Département

Ce widget permet d'analyser simultanément **la sécurité** et **le coût du logement** dans chaque département d'Île-de-France.

L'utilisateur peut observer :
- Un **graphique de corrélation** entre le taux de criminalité et le prix moyen des logements,
- Un **classement des départements** selon un score global combinant sécurité et accessibilité financière,
- Une **normalisation des scores** sur une échelle de 0 à 10 pour faciliter la comparaison.

Le système calcule :
- Un **score de sécurité** inversement proportionnel au taux de criminalité,
- Un **score de prix** inversement proportionnel au coût moyen des logements,
- Un **score global** représentant la moyenne pondérée des deux indicateurs.

Les résultats sont présentés sous forme de :
- **Scatter plot** montrant la relation entre criminalité et prix (taille des points proportionnelle au score),
- **Graphique en barres horizontales** classant les départements par score global,
- **Tableau récapitulatif** détaillant les trois meilleurs et les trois moins favorables départements.

Ce module aide l'étudiant à identifier les départements offrant le meilleur équilibre entre sécurité et budget, permettant ainsi un choix de localisation éclairé.

In [ ]:
# Préparation des données CROUS
crous_clean = crous.copy()
crous_clean['prix'] = smart_to_float(crous_clean['loyer_mediane_eur'])
crous_clean['arrondissement'] = crous_clean['arrondissement'].fillna(0).astype(int).astype(str)
crous_clean['departement'] = crous_clean['arrondissement'].apply(
    lambda x: '75' if x.startswith('75') or (x.isdigit() and int(x) < 20) else x[:2] if len(x) >= 2 else '75'
)
crous_clean = crous_clean[crous_clean['prix'] > 0]

# Préparation des données logements privés
noncrous_clean = noncrous.copy()
noncrous_clean['prix'] = smart_to_float(noncrous_clean['loyer'])
noncrous_clean['departement'] = noncrous_clean['arrondissement'].apply(extract_dept_from_cp)
noncrous_clean = noncrous_clean[noncrous_clean['prix'] > 0]

# Display dataframes before concatenation for debugging
print("crous_clean columns:", crous_clean.columns)
display(crous_clean.head())
print("\nnoncrous_clean columns:", noncrous_clean.columns)
display(noncrous_clean.head())


# Préparation données criminalité
crime_clean = crime.copy()
crime_clean['Code_departement'] = crime_clean['Code_departement'].astype(str)

# Calcul des prix moyens par département (CROUS + Privé)
prix_by_dept = pd.concat([
    crous_clean[['departement', 'prix']],
    noncrous_clean[['departement', 'prix']]
]).groupby('departement')['prix'].mean().reset_index()

# Fusion avec criminalité
securite_budget = crime_clean.merge(
    prix_by_dept,
    left_on='Code_departement',
    right_on='departement',
    how='left'
)

securite_budget = securite_budget[securite_budget['prix'].notna()]

# Normalisation des scores (0-10)
securite_budget['score_securite'] = 10 - (securite_budget['taux_pour_1000_hab'] / securite_budget['taux_pour_1000_hab'].max() * 10)
securite_budget['score_prix'] = 10 - (securite_budget['prix'] / securite_budget['prix'].max() * 10)
securite_budget['score_global'] = (securite_budget['score_securite'] + securite_budget['score_prix']) / 2
securite_budget = securite_budget.sort_values('score_global', ascending=False)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique 1 : Scatter plot Sécurité vs Prix
scatter = axes[0].scatter(securite_budget['taux_pour_1000_hab'],
                          securite_budget['prix'],
                          s=securite_budget['score_global']*50,
                          c=securite_budget['score_global'],
                          cmap='RdYlGn',
                          alpha=0.7,
                          edgecolors='black',
                          linewidth=1)

for idx, row in securite_budget.iterrows():
    axes[0].annotate(row['Code_departement'],
                     (row['taux_pour_1000_hab'], row['prix']),
                     fontsize=10, fontweight='bold')

axes[0].set_xlabel('Taux de criminalité (pour 1000 habitants)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Prix moyen des logements (euros)', fontsize=12, fontweight='bold')
axes[0].set_title('Corrélation Sécurité et Budget par Département', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0], label='Score Global sur 10')

# Graphique 2 : Classement des départements
colors = ['#22c55e' if s > 7 else '#fb923c' if s > 5 else '#ef4444'
          for s in securite_budget['score_global']]
axes[1].barh(securite_budget['Code_departement'],
             securite_budget['score_global'],
             color=colors, alpha=0.8)
axes[1].set_xlabel('Score Global sur 10', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Département', fontsize=12, fontweight='bold')
axes[1].set_title('Classement des Départements par Score Sécurité-Budget', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Tableau récapitulatif
print("\nTop 3 des Meilleurs Départements (Sécurité + Budget) :")
print("="*80)
for i, row in securite_budget.head(3).iterrows():
    print(f"  Département {row['Code_departement']} - Score: {row['score_global']:.2f}/10")
    print(f"    Taux de criminalité: {row['taux_pour_1000_hab']:.1f} pour 1000 habitants")
    print(f"    Prix moyen: {row['prix']:.0f} euros")
    print()

print("Top 3 des Départements Moins Favorables :")
print("="*80)
for i, row in securite_budget.tail(3).iterrows():
    print(f"  Département {row['Code_departement']} - Score: {row['score_global']:.2f}/10")
    print(f"    Taux de criminalité: {row['taux_pour_1000_hab']:.1f} pour 1000 habitants")
    print(f"    Prix moyen: {row['prix']:.0f} euros")
    print()

---
## *Widget 8* : Score d'Accessibilité par École

Ce widget calcule un **indice d'accessibilité composite** pour chaque établissement d'enseignement supérieur en Île-de-France.

L'utilisateur peut :
- Choisir entre afficher le **Top 10** des écoles les mieux desservies ou le **Flop 10** des écoles les moins accessibles,
- Consulter pour chaque école un **score global sur 10** ainsi que les détails de calcul.

Le système évalue trois critères principaux :
- **Proximité des transports** : nombre de gares situées à moins d'un kilomètre de l'école,
- **Offre de logements** : nombre de logements (CROUS et privés) disponibles dans un rayon de trois kilomètres,
- **Accessibilité financière** : prix moyen des logements à proximité.

Le calcul du score se décompose en :
- **40% pour la proximité des transports** : plus il y a de gares proches, meilleur est le score,
- **30% pour l'offre de logements** : un nombre élevé de logements augmente les chances de trouver un hébergement,
- **30% pour le prix moyen** : un prix bas améliore le score d'accessibilité financière.

Les résultats sont présentés sous forme de :
- **Graphique en barres horizontales** classant les écoles par score (code couleur : vert si excellent et rouge si faible),
- **Tableau détaillé** indiquant pour chaque établissement le nombre de gares proches, le nombre de logements disponibles et le prix moyen.

Ce module permet d'identifier rapidement les établissements offrant les meilleures conditions d'accessibilité et de logement pour les étudiants.

In [ ]:
# Préparation des coordonnées écoles
ecoles_clean = ecoles.copy()
ecoles_clean['latitude'] = smart_to_float(ecoles_clean['lat'])
ecoles_clean['longitude'] = smart_to_float(ecoles_clean['lon'])
ecoles_clean = ecoles_clean.dropna(subset=['latitude', 'longitude'])

# Extraction coordonnées gares
gares_clean = gares.copy()
gares_clean['coords'] = gares_clean['Geo Point'].str.split(', ')
gares_clean['latitude'] = gares_clean['coords'].apply(
    lambda x: float(x[0]) if isinstance(x, list) and len(x) >= 2 else np.nan
)
gares_clean['longitude'] = gares_clean['coords'].apply(
    lambda x: float(x[1]) if isinstance(x, list) and len(x) >= 2 else np.nan
)
gares_clean = gares_clean.dropna(subset=['latitude', 'longitude'])

# Préparation logements (CROUS + Privés)
print("Combining housing data manually...")
all_logements_list = []

# Add CROUS data
for idx, row in crous_clean.iterrows():
    if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude')) and pd.notna(row.get('prix')):
        all_logements_list.append({
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'prix': row['prix']
        })

# Add non-CROUS data
for idx, row in noncrous_clean.iterrows():
     if pd.notna(row.get('latitude')) and pd.notna(row.get('longitude')) and pd.notna(row.get('prix')):
        all_logements_list.append({
            'latitude': row['latitude'],
            'longitude': row['longitude'],
            'prix': row['prix']
        })

all_logements = pd.DataFrame(all_logements_list)

all_logements['latitude'] = smart_to_float(all_logements['latitude'])
all_logements['longitude'] = smart_to_float(all_logements['longitude'])
all_logements = all_logements.dropna(subset=['latitude', 'longitude'])

print(f"Finished combining data. Total housing points: {len(all_logements)}")


# Calcul des scores pour chaque école
print("Calculating accessibility scores for each school...")
scores_data = []

for idx, ecole in ecoles_clean.iterrows():
    e_lat, e_lon = ecole['latitude'], ecole['longitude']
    e_nom = ecole["nom"] # Use the correct column name 'nom'

    # Nombre de gares à moins de 1km
    gares_clean['dist'] = dist_km(gares_clean['latitude'], gares_clean['longitude'], e_lat, e_lon)
    nb_gares_proches = (gares_clean['dist'] <= 1).sum()

    # Nombre de logements à moins de 3km
    all_logements['dist'] = dist_km(all_logements['latitude'], all_logements['longitude'], e_lat, e_lon)
    logements_proches = all_logements[all_logements['dist'] <= 3]
    nb_logements = len(logements_proches)
    prix_moyen = logements_proches['prix'].mean() if nb_logements > 0 else np.nan

    # Calcul du score (sur 10)
    score_gares = min(nb_gares_proches * 1.5, 10)
    score_logements = min(nb_logements / 50 * 10, 10)
    score_prix = (1 - (prix_moyen / 2000)) * 10 if not pd.isna(prix_moyen) else 5
    score_prix = max(0, min(score_prix, 10))

    score_global = (score_gares * 0.4 + score_logements * 0.3 + score_prix * 0.3)

    scores_data.append({
        'ecole': e_nom,
        'nb_gares_1km': nb_gares_proches,
        'nb_logements_3km': nb_logements,
        'prix_moyen': prix_moyen,
        'score_gares': score_gares,
        'score_logements': score_logements,
        'score_prix': score_prix,
        'score_global': score_global
    })

scores_df = pd.DataFrame(scores_data).sort_values('score_global', ascending=False)
print("Calculation finished.\n")

# Widget interactive
def afficher_score_ecole(mode='Top 10'):
    if mode == 'Top 10':
        data = scores_df.head(10)
        titre = 'Top 10 Écoles - Meilleure Accessibilité'
    else:
        data = scores_df.tail(10).iloc[::-1]
        titre = 'Top 10 Écoles - Accessibilité Faible'

    fig, ax = plt.subplots(figsize=(14, 8))

    colors = ['#22c55e' if s > 7 else '#fb923c' if s > 5 else '#ef4444'
              for s in data['score_global']]

    y_pos = np.arange(len(data))
    ax.barh(y_pos, data['score_global'], color=colors, alpha=0.8, edgecolor='black', linewidth=1)

    ax.set_yticks(y_pos)
    ax.set_yticklabels([nom[:50] + '...' if len(nom) > 50 else nom for nom in data['ecole']], fontsize=9)
    ax.set_xlabel('Score d\'Accessibilité sur 10', fontsize=12, fontweight='bold')
    ax.set_title(titre, fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

    # Adding values
    for i, v in enumerate(data['score_global']):
        ax.text(v + 0.2, i, f'{v:.1f}', va='center', fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.show()

    # Details
    print(f"\nDetails des écoles :")
    print("="*100)
    for idx, row in data.iterrows():
        print(f"\nÉcole : {row['ecole'][:70]}")
        print(f"  Score Global: {row['score_global']:.1f}/10")
        print(f"  Gares à moins de 1km: {row['nb_gares_1km']}")
        print(f"  Logements à moins de 3km: {row['nb_logements_3km']}")
        if not pd.isna(row['prix_moyen']):
            print(f"  Prix moyen des logements: {row['prix_moyen']:.0f} euros")
        else:
            print(f"  Prix moyen: Non disponible")


interact(afficher_score_ecole,
         mode=Dropdown(options=['Top 10', 'Flop 10'], value='Top 10', description='Affichage:'));

---
## *Widget 9* : Simulateur de Budget Mensuel Étudiant

Ce widget permet de **simuler et visualiser le budget mensuel complet** d'un étudiant en fonction de son choix de logement et de son école.

L'utilisateur peut :
- Sélectionner une **école** dans la liste des établissements d'Île-de-France,
- Définir un **budget maximum** pour le loyer,
- Ajuster une **estimation des dépenses alimentaires** mensuelles,
- Préciser un **budget transport** mensuel (pass Navigo, vélo, etc.).

Le système calcule automatiquement :
- Le **loyer moyen** des logements (CROUS et privés) situés à proximité de l'école sélectionnée,
- La **répartition budgétaire** entre loyer, alimentation, transport et autres dépenses,
- Le **budget total mensuel estimé** pour vivre en tant qu'étudiant dans cette zone.

Les résultats sont présentés sous forme de :
- **Graphique circulaire** (camembert) montrant la proportion de chaque poste de dépense,
- **Graphique en barres** détaillant le montant exact de chaque catégorie budgétaire,
- **Tableau récapitulatif** affichant le budget total et le détail des dépenses.

Ce module aide l'étudiant à **anticiper ses dépenses mensuelles** et à évaluer la faisabilité financière de son projet d'études en fonction de la localisation choisie.

In [ ]:
def simulateur_budget(ecole, budget_loyer_max=800, budget_alimentation=250, budget_transport=75):
    """
    Simule le budget mensuel d'un étudiant en fonction de son école.
    """

    # Trouver l'école
    row_ecole = ecoles_clean[ecoles_clean["nom"] == ecole]
    if row_ecole.empty:
        print("École non trouvée.")
        return

    e_lat, e_lon = row_ecole.iloc[0]['latitude'], row_ecole.iloc[0]['longitude']

    # Calculer le loyer moyen des logements proches (rayon 5km)
    all_logements['dist_ecole'] = dist_km(all_logements['latitude'], all_logements['longitude'], e_lat, e_lon)
    logements_proches = all_logements[
        (all_logements['dist_ecole'] <= 5) &
        (all_logements['prix'] <= budget_loyer_max) &
        (all_logements['prix'] > 0)
    ]

    if logements_proches.empty:
        print(f"Aucun logement trouvé dans un rayon de 5km avec un budget de {budget_loyer_max} euros.")
        return

    loyer_moyen = logements_proches['prix'].mean()
    loyer_median = logements_proches['prix'].median()
    nb_logements_dispo = len(logements_proches)

    # Calcul du budget total
    budget_autres = 150  # Loisirs, téléphone, etc.
    budget_total = loyer_median + budget_alimentation + budget_transport + budget_autres

    # Préparation des données pour visualisation
    categories = ['Loyer', 'Alimentation', 'Transport', 'Autres']
    montants = [loyer_median, budget_alimentation, budget_transport, budget_autres]
    couleurs = ['#3b82f6', '#22c55e', '#fb923c', '#a855f7']

    # Visualisation
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Graphique 1 : Camembert
    axes[0].pie(montants, labels=categories, autopct='%1.1f%%',
                colors=couleurs, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
    axes[0].set_title(f'Répartition du Budget Mensuel\nTotal: {budget_total:.0f} euros',
                      fontsize=14, fontweight='bold')

    # Graphique 2 : Barres
    axes[1].bar(categories, montants, color=couleurs, alpha=0.8, edgecolor='black', linewidth=1.5)
    axes[1].set_ylabel('Montant (euros)', fontsize=12, fontweight='bold')
    axes[1].set_title('Détail des Dépenses Mensuelles', fontsize=14, fontweight='bold')
    axes[1].grid(axis='y', alpha=0.3)

    # Ajout des valeurs sur les barres
    for i, v in enumerate(montants):
        axes[1].text(i, v + 10, f'{v:.0f}€', ha='center', fontweight='bold', fontsize=11)

    plt.tight_layout()
    plt.show()

    # Récapitulatif textuel
    print("\nRécapitulatif du Budget Mensuel Étudiant")
    print("="*70)
    print(f"École sélectionnée : {ecole}")
    print(f"Nombre de logements disponibles (rayon 5km) : {nb_logements_dispo}")
    print()
    print("Détail des dépenses :")
    print(f"  - Loyer médian : {loyer_median:.0f} euros (moyenne : {loyer_moyen:.0f} euros)")
    print(f"  - Alimentation : {budget_alimentation:.0f} euros")
    print(f"  - Transport : {budget_transport:.0f} euros")
    print(f"  - Autres dépenses : {budget_autres:.0f} euros")
    print()
    print(f"Budget Total Mensuel : {budget_total:.0f} euros")
    print(f"Budget Annuel Estimé : {budget_total * 12:.0f} euros (hors vacances)")
    print()

    # Conseils
    if budget_total < 1000:
        print("Observation : Budget très raisonnable pour un étudiant.")
    elif budget_total < 1300:
        print("Observation : Budget moyen, nécessite une gestion attentive.")
    else:
        print("Observation : Budget élevé, recherche de solutions d'économie recommandée.")

# Interface
interact(simulateur_budget,
         ecole=Dropdown(
             options=sorted(ecoles_clean["nom"].unique()),
             description='École :',
             style={'description_width': 'initial'},
             layout=Layout(width='500px')
         ),
         budget_loyer_max=IntSlider(
             value=800, min=300, max=1500, step=50,
             description='Budget loyer max (euros) :',
             style={'description_width': 'initial'}
         ),
         budget_alimentation=IntSlider(
             value=250, min=150, max=500, step=25,
             description='Budget alimentation (euros) :',
             style={'description_width': 'initial'}
         ),
         budget_transport=IntSlider(
             value=75, min=0, max=150, step=25,
             description='Budget transport (euros) :',
             style={'description_width': 'initial'}
         ));


## *Widget 10* : Carte de Densité des Services Étudiants

Ce widget génère une **carte interactive avec heatmap** visualisant la concentration géographique des services essentiels pour les étudiants en Île-de-France.

L'utilisateur peut :
- Choisir le **type de services** à afficher : tous les services combinés, uniquement les logements, uniquement les écoles, ou uniquement les espaces verts,
- Ajuster le **rayon d'influence** de la heatmap pour modifier la granularité de la visualisation,
- Modifier le **niveau de zoom** pour explorer différentes échelles géographiques.

La carte affiche :
- Une **heatmap colorée** indiquant la densité des services (bleu pour faible, vert pour moyenne, jaune à rouge pour forte),
- Des **marqueurs spécifiques** selon le type sélectionné :
  - Icônes bleues pour les écoles avec tooltip affichant le nom,
  - Points rouges pour les gares principales,
  - Zones colorées pour les logements et espaces verts.

Le système analyse :
- Les **logements CROUS** avec leurs coordonnées géographiques,
- Un **échantillon représentatif** de logements privés (500 maximum pour optimiser les performances),
- L'ensemble des **établissements d'enseignement supérieur**,
- Un échantillon d'**espaces verts** (200 maximum),
- Un échantillon de **gares** (50 maximum).

Les résultats sont présentés sous forme de :
- **Carte interactive Folium** permettant le zoom et la navigation,
- **Légende intégrée** expliquant le code couleur de la densité,
- **Statistiques détaillées** indiquant le nombre exact de points analysés pour chaque catégorie.

Ce module permet d'identifier visuellement les **zones les mieux équipées** en services étudiants, facilitant ainsi le choix d'une localisation optimale en fonction des priorités de chacun.

In [ ]:
from folium.plugins import HeatMap

def carte_densite_services(type_service='Tous', rayon_heatmap=15, zoom=11):
    noncrous.columns = noncrous.columns.str.strip().str.lower()
    for col in noncrous.columns:
        if re.search("lat|lon|lng", col, re.I):
            noncrous[col] = pd.to_numeric(noncrous[col], errors="coerce")
    for col in gares_clean.columns:
        if re.search("lat|lon|lng", col, re.I):
            gares_clean[col] = pd.to_numeric(gares_clean[col], errors="coerce")
    gares_clean.dropna(subset=[c for c in gares_clean.columns if re.search("lat|lon", c, re.I)], inplace=True)


    def detect_latlon(df, df_name="DataFrame"):
        lat_col = next((c for c in df.columns if re.search("lat", c, re.I)), None)
        lon_col = next((c for c in df.columns if re.search("lon|lng", c, re.I)), None)
        if not lat_col or not lon_col:
            print(f" Avertissement : colonnes latitude/longitude introuvables dans {df_name}")
        return lat_col, lon_col

    lat_c, lon_c = detect_latlon(crous_clean, "crous_clean")
    lat_n, lon_n = detect_latlon(noncrous, "noncrous")
    lat_e, lon_e = detect_latlon(ecoles_clean, "ecoles_clean")
    lat_p, lon_p = detect_latlon(parks, "parks")
    lat_g, lon_g = detect_latlon(gares_clean, "gares_clean")

    m = folium.Map(location=[48.8566, 2.3522], zoom_start=zoom, tiles='OpenStreetMap')
    points = []

    if type_service in ['Tous', 'Logements']:
        if lat_c and lon_c:
            for _, row in crous_clean.iterrows():
                if pd.notna(row[lat_c]) and pd.notna(row[lon_c]):
                    points.append([row[lat_c], row[lon_c]])

        if lat_n and lon_n:
            sample_noncrous = noncrous.sample(min(500, len(noncrous)))
            for _, row in sample_noncrous.iterrows():
                if pd.notna(row[lat_n]) and pd.notna(row[lon_n]):
                    points.append([row[lat_n], row[lon_n]])
        else:
            print(" Aucun point Non-CROUS ajouté (colonnes manquantes).")

    if type_service in ['Tous', 'Écoles'] and lat_e and lon_e:
        for _, row in ecoles_clean.iterrows():
            if pd.notna(row[lat_e]) and pd.notna(row[lon_e]):
                points.append([row[lat_e], row[lon_e]])
                if type_service == 'Écoles':
                    folium.Marker(
                        [row[lat_e], row[lon_e]],
                        popup=row.get("nom", "École"),
                        icon=folium.Icon(color='blue', icon='graduation-cap', prefix='fa'),
                        tooltip=row.get("nom", "")[:50]
                    ).add_to(m)

    if type_service in ['Tous', 'Espaces verts'] and lat_p and lon_p:
        parks_sample = parks.sample(min(200, len(parks)))
        for _, row in parks_sample.iterrows():
            if pd.notna(row[lat_p]) and pd.notna(row[lon_p]):
                points.append([row[lat_p], row[lon_p]])
    if points:
        HeatMap(
            points,
            radius=rayon_heatmap,
            blur=rayon_heatmap + 5,
            max_zoom=13,
            gradient={
                0.0: 'blue',
                0.3: 'cyan',
                0.5: 'lime',
                0.7: 'yellow',
                1.0: 'red'
            }
        ).add_to(m)
    else:
        print(" Aucun point ajouté à la carte — vérifie les colonnes latitude/longitude.")


    if type_service == 'Tous' and lat_g and lon_g:
        gares_sample = gares_clean.sample(min(50, len(gares_clean)))
        for _, row in gares_sample.iterrows():
            if pd.notna(row[lat_g]) and pd.notna(row[lon_g]):
                folium.CircleMarker(
                    [row[lat_g], row[lon_g]],
                    radius=3,
                    popup=row.get('nom_long', 'Gare'),
                    color='red',
                    fill=True,
                    fillColor='red',
                    fillOpacity=0.7
                ).add_to(m)


    legend_html = f'''
    <div style="
        position: fixed;
        bottom: 50px; left: 50px; width: 280px;
        background-color: white;
        border: 2px solid grey;
        border-radius: 5px;
        padding: 10px;
        font-size: 14px;
        z-index: 9999;
    ">
    <p style="margin:0; font-weight:bold;">Densité des Services : {type_service}</p>
    <p style="margin:5px 0;"><span style="color:blue;">■</span> Faible densité</p>
    <p style="margin:5px 0;"><span style="color:lime;">■</span> Densité moyenne</p>
    <p style="margin:5px 0;"><span style="color:red;">■</span> Forte densité</p>
    <p style="margin:5px 0; font-size:12px;">Note : Les zones rouges indiquent les meilleures zones</p>
    <p style="margin:5px 0; font-size:11px;">Nombre de points analysés : {len(points)}</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

    display(m)


    print(f"\nStatistiques pour le type : {type_service}")
    print("=" * 60)
    print(f"Nombre total de points analysés : {len(points)}")
    if type_service in ['Tous', 'Logements']:
        print(f"  Logements CROUS : {len(crous_clean)}")
        print(f"  Logements privés (échantillon) : {min(500, len(noncrous))}")
    if type_service in ['Tous', 'Écoles']:
        print(f"  Établissements d'enseignement : {len(ecoles_clean)}")
    if type_service in ['Tous', 'Espaces verts']:
        print(f"  Espaces verts (échantillon) : {min(200, len(parks))}")
    print("\nConclusion : Les zones colorées en rouge présentent la plus forte")
    print("concentration de services et sont donc les plus adaptées pour les étudiants.")



interact(
    carte_densite_services,
    type_service=Dropdown(
        options=['Tous', 'Logements', 'Écoles', 'Espaces verts'],
        value='Tous',
        description='Type de services :',
        style={'description_width': 'initial'}
    ),
    rayon_heatmap=IntSlider(
        value=15, min=5, max=30, step=5,
        description='Rayon heatmap :',
        style={'description_width': 'initial'}
    ),
    zoom=IntSlider(
        value=11, min=9, max=13, step=1,
        description='Niveau de zoom :',
        style={'description_width': 'initial'}
    )
)


In [ ]:
print(noncrous.columns.tolist())
